In [6]:
%pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 284.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/9.7 MB 284.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/9.7 MB 284.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/9.7 MB 284.3 kB/s eta 0:00:33
   -- ------------------------------------- 0.5/9.7 MB 284.3 kB/s eta 0:00:33
   --- ------------------------------------ 0.8/9.7 MB 268

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import sys
import importlib.util

print("当前 Python：", sys.executable)
print("能否找到 TensorFlow：",
      importlib.util.find_spec("tensorflow") is not None)

当前 Python： C:\Users\MSI-NB\.conda\envs\gpu\python.exe
能否找到 TensorFlow： True


In [3]:
print("开始导入 TensorFlow……", flush=True)

import tensorflow as tf

print("TensorFlow 版本：", tf.__version__)
print("GPU：", tf.config.list_physical_devices("GPU"))

开始导入 TensorFlow……
TensorFlow 版本： 2.10.0
GPU： [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
from pathlib import Path
data_root = (
    Path.cwd()
    / "../../data/LG 18650HG2 Li-ion Battery Data"
    / "LG_HG2_Original_Dataset_McMasterUniversity_Jan_2020"
).resolve()

print("当前目录：", Path.cwd())
print("数据目录：", data_root)
print("目录存在：", data_root.is_dir())

for temp in ("0degC", "10degC", "25degC"):
    print(temp, "CSV 数量：", len(list((data_root / temp).glob("*.csv"))))

当前目录： C:\Users\MSI-NB\研究生\Deep Lstm\battery-state-estimation\experiments\lg
数据目录： C:\Users\MSI-NB\研究生\Deep Lstm\battery-state-estimation\data\LG 18650HG2 Li-ion Battery Data\LG_HG2_Original_Dataset_McMasterUniversity_Jan_2020
目录存在： True
0degC CSV 数量： 32
10degC CSV 数量： 32
25degC CSV 数量： 36


# Main notebook for battery state estimation

In [1]:
import numpy as np
import pandas as pd
import scipy.io
import math
import os
import ntpath
import sys
import logging
import time
import sys

from importlib import reload
import plotly.graph_objects as go

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation
from keras.optimizers import SGD, Adam
from keras.utils import np_utils
from keras.layers import LSTM, Embedding, RepeatVector, TimeDistributed, Masking
from keras.callbacks import EarlyStopping, ModelCheckpoint, LambdaCallback


IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = "/content/drive/My Drive/battery-state-estimation/battery-state-estimation/"
else:
    data_path = "../../"

sys.path.append(data_path)
from data_processing.lg_dataset import LgData

In [2]:
print("TensorFlow 版本：", tf.__version__)
print("GPU：", tf.config.list_physical_devices("GPU"))

TensorFlow 版本： 2.10.0
GPU： [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Config logging

In [3]:
reload(logging)
logging.basicConfig(format='%(asctime)s [%(levelname)s]: %(message)s', level=logging.DEBUG, datefmt='%Y/%m/%d %H:%M:%S')

# Load Data

In [4]:
train_names = [
    '0degC/589_Mixed1',
    '0degC/589_Mixed2',
    '0degC/590_Mixed4',
    '0degC/590_Mixed5',
    '0degC/590_Mixed6',
    '0degC/590_Mixed8',

    '10degC/567_Mixed1',
    '10degC/567_Mixed2',
    '10degC/571_Mixed4',
    '10degC/571_Mixed5',
    '10degC/571_Mixed6',
    '10degC/571_Mixed8',
    
    '25degC/551_Mixed1', 
    '25degC/551_Mixed2', 
    #'25degC/552_Mixed3',
    '25degC/552_Mixed4', 
    '25degC/552_Mixed5', 
    '25degC/552_Mixed6', 
    '25degC/552_Mixed8',   
    ]
test_names = [
    '0degC/589_LA92',
    '0degC/589_UDDS',
    '0degC/589_US06',
    '0degC/590_Mixed7',    
        
    '10degC/582_LA92',
    '10degC/576_UDDS',
    '10degC/567_US06',
    '10degC/571_Mixed7',

    '25degC/551_LA92', 
    '25degC/551_UDDS', 
    '25degC/551_US06',
    '25degC/552_Mixed7', 
    ]

steps = 300

lg_data = LgData(data_path)
cycles = lg_data.get_discharge_whole_cycle(train_names, test_names, output_capacity=False, scale_test=True)
train_x, train_y, test_x, test_y = lg_data.get_discharge_multiple_step(cycles, steps)

train_y = lg_data.keep_only_y_end(train_y, steps)
test_y = lg_data.keep_only_y_end(test_y, steps)

2026/10/25 20:55:07 [INFO]: There is a NaN in cycle 10degC/582_LA92, removing row
2026/10/25 20:55:36 [INFO]: Train x: (4068, 300, 3), train y: (4068, 300, 1) | Test x: (3218, 300, 3), test y: (3218, 300, 1)


In [5]:
print("train_x / train_y：", train_x.shape, train_y.shape)
print("test_x / test_y：", test_x.shape, test_y.shape)

train_x / train_y： (4068, 300, 3) (4068, 1, 1)
test_x / test_y： (3218, 300, 3) (3218, 1, 1)


In [6]:
print("四个数组全部为有限数：",
      all(np.isfinite(a).all()
          for a in (train_x, train_y, test_x, test_y)))

print("训练 SOC 范围：", train_y.min(), train_y.max())
print("测试 SOC 范围：", test_y.min(), test_y.max())

bad_alignment = [
    (name, len(x), len(y))
    for names, group in [(train_names, cycles[0]), (test_names, cycles[1])]
    for name, (x, y) in zip(names, group)
    if len(x) != len(y)
]
print("输入/标签行数异常：", bad_alignment)

print("TensorFlow 版本：", tf.__version__)
print("可见 GPU：", tf.config.list_physical_devices("GPU"))

四个数组全部为有限数： True
训练 SOC 范围： 5.106404708100799e-05 0.9993276759085169
测试 SOC 范围： 0.00028934945974328294 0.9999171852937645
输入/标签行数异常： []
TensorFlow 版本： 2.10.0
可见 GPU： [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Model training

In [7]:
from pathlib import Path

model_dir = (Path(data_path) / "results" / "trained_model").resolve()
model_dir.mkdir(parents=True, exist_ok=True)
print("模型保存目录：", model_dir)

模型保存目录： C:\Users\MSI-NB\研究生\Deep Lstm\battery-state-estimation\results\trained_model


In [8]:
EXPERIMENT = "lstm_soc_percentage_lg_positive_temp_300_steps_mixed_cycle_test"

experiment_name = time.strftime("%Y-%m-%d-%H-%M-%S") + '_' + EXPERIMENT
print(experiment_name)

#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation='selu',
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation='selu', return_sequences=False))
model.add(Dense(256, activation='selu'))
model.add(Dense(128, activation='selu'))
model.add(Dense(1, activation='linear'))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=50)
mc = ModelCheckpoint(data_path + 'results/trained_model/%s_best.h5' % experiment_name, 
                             save_best_only=True, 
                             monitor='val_loss')
print("模型参数量：", model.count_params())
print("第一个权重所在设备：", model.weights[0].device)

2026-10-25-20-55-49_lstm_soc_percentage_lg_positive_temp_300_steps_mixed_cycle_test


C:\Users\MSI-NB\.conda\envs\gpu\lib\site-packages\keras\optimizers\optimizer_v2\adam.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 300, 256)          266240    
                                                                 
 lstm_1 (LSTM)               (None, 256)               525312    
                                                                 
 dense (Dense)               (None, 256)               65792     
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dense_2 (Dense)             (None, 1)                 129       
                                                                 
Total params: 890,369
Trainable params: 890,369
Non-trainable params: 0
_________________________________________________________________
模型参数量： 890369
第一个权重所在设备： /job:localhost/replica:0/

In [9]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=32, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

Epoch 1/1000
102/102 - 104s - loss: 0.0341 - mse: 0.0689 - mae: 0.1549 - mape: 118.2183 - rmse: 0.2626 - val_loss: 0.0024 - val_mse: 0.0047 - val_mae: 0.0550 - val_mape: 47.4547 - val_rmse: 0.0686 - 104s/epoch - 1s/step
Epoch 2/1000
102/102 - 103s - loss: 0.0013 - mse: 0.0025 - mae: 0.0381 - mape: 86.6597 - rmse: 0.0502 - val_loss: 0.0013 - val_mse: 0.0025 - val_mae: 0.0389 - val_mape: 35.0592 - val_rmse: 0.0502 - 103s/epoch - 1s/step
Epoch 3/1000
102/102 - 107s - loss: 9.5651e-04 - mse: 0.0019 - mae: 0.0333 - mape: 70.3873 - rmse: 0.0437 - val_loss: 8.8705e-04 - val_mse: 0.0018 - val_mae: 0.0323 - val_mape: 28.9010 - val_rmse: 0.0421 - 107s/epoch - 1s/step
Epoch 4/1000
102/102 - 105s - loss: 8.0373e-04 - mse: 0.0016 - mae: 0.0306 - mape: 67.5365 - rmse: 0.0401 - val_loss: 7.5010e-04 - val_mse: 0.0015 - val_mae: 0.0299 - val_mape: 26.2301 - val_rmse: 0.0387 - 105s/epoch - 1s/step
Epoch 5/1000
102/102 - 102s - loss: 7.2282e-04 - mse: 0.0014 - mae: 0.0292 - mape: 49.7318 - rmse: 0.0380 -

In [10]:
model.save(data_path + 'results/trained_model/%s.h5' % experiment_name)

hist_df = pd.DataFrame(history.history)
hist_csv_file = data_path + 'results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

### Testing

In [11]:
results = model.evaluate(test_x, test_y)
print(results)

101/101 [==============================] - 24s 231ms/step - loss: 2.0705e-04 - mse: 4.1411e-04 - mae: 0.0153 - mape: 16.6706 - rmse: 0.0203
[0.0002070540504064411, 0.0004141081008128822, 0.015347233973443508, 16.670618057250977, 0.020349645987153053]


In [12]:
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error
)

# 使用作者原方法，准备 25°C 四条测试工况
test_names_25 = [
    name for name in test_names if name.startswith("25degC/")
]

cycles_25 = lg_data.get_discharge_whole_cycle(
    train_names, test_names_25,
    output_capacity=False,
    scale_test=True
)

_, _, test_x_25, test_y_25 = lg_data.get_discharge_multiple_step(
    cycles_25, steps
)
test_y_25 = lg_data.keep_only_y_end(test_y_25, steps)

print("25°C 测试文件：", test_names_25)
print("25°C 输入形状：", test_x_25.shape)
assert test_x_25.shape == (1164, 300, 3)

# 真实 SOC 与预测 SOC 一一配对
y_true_25 = test_y_25[:, :, 0].reshape(-1)
y_pred_25 = model.predict(test_x_25, verbose=0).reshape(-1)
assert y_true_25.shape == y_pred_25.shape

print("25°C MAE（百分点）：",
      100 * mean_absolute_error(y_true_25, y_pred_25))
print("25°C RMSE（百分点）：",
      100 * np.sqrt(mean_squared_error(y_true_25, y_pred_25)))
print("25°C R²：", r2_score(y_true_25, y_pred_25))

# 保存成对数值，后续可以重新计算指标
prediction_path = model_dir / f"{experiment_name}_25degC_predictions.csv"
pd.DataFrame({
    "SOC_true": y_true_25,
    "SOC_pred": y_pred_25
}).to_csv(prediction_path, index=False)

print("预测结果保存位置：", prediction_path)

2026/10/26 09:08:16 [INFO]: Train x: (4068, 300, 3), train y: (4068, 300, 1) | Test x: (1164, 300, 3), test y: (1164, 300, 1)


25°C 测试文件： ['25degC/551_LA92', '25degC/551_UDDS', '25degC/551_US06', '25degC/552_Mixed7']
25°C 输入形状： (1164, 300, 3)
25°C MAE（百分点）： 1.360534027718608
25°C RMSE（百分点）： 1.7528900472015903
25°C R²： 0.9960310520451567
预测结果保存位置： C:\Users\MSI-NB\研究生\Deep Lstm\battery-state-estimation\results\trained_model\2026-10-25-20-55-49_lstm_soc_percentage_lg_positive_temp_300_steps_mixed_cycle_test_25degC_predictions.csv


# Data Visualization

In [ ]:
# fig = go.Figure()
# fig.add_trace(go.Scatter(y=history.history['loss'],
#                     mode='lines', name='train'))
# fig.add_trace(go.Scatter(y=history.history['val_loss'],
#                     mode='lines', name='validation'))
# fig.update_layout(title='Loss trend',
#                   xaxis_title='epoch',
#                   yaxis_title='loss')
# fig.show()

In [ ]:
# train_predictions = model.predict(train_x)

In [ ]:
# cycle_num = 0
# steps_num = 8000
# step_index = np.arange(cycle_num*steps_num, (cycle_num+1)*steps_num)

# fig = go.Figure()
# fig.add_trace(go.Scatter(x=step_index, y=train_predictions.flatten()[cycle_num*steps_num:(cycle_num+1)*steps_num],
#                     mode='lines', name='SoC predicted'))
# fig.add_trace(go.Scatter(x=step_index, y=train_y.flatten()[cycle_num*steps_num:(cycle_num+1)*steps_num],
#                     mode='lines', name='SoC actual'))
# fig.update_layout(title='Results on training',
#                   xaxis_title='Step',
#                   yaxis_title='SoC percentage')
# fig.show()

In [ ]:
# test_predictions = model.predict(test_x)

In [ ]:
# cycle_num = 0
# steps_num = 8000
# step_index = np.arange(cycle_num*steps_num, (cycle_num+1)*steps_num)

# fig = go.Figure()
# fig.add_trace(go.Scatter(x=step_index, y=test_predictions.flatten()[cycle_num*steps_num:(cycle_num+1)*steps_num],
#                     mode='lines', name='SoC predicted'))
# fig.add_trace(go.Scatter(x=step_index, y=test_y.flatten()[cycle_num*steps_num:(cycle_num+1)*steps_num],
#                     mode='lines', name='SoC actual'))
# fig.update_layout(title='Results on testing',
#                   xaxis_title='Step',
#                   yaxis_title='SoC percentage')
# fig.show()